In [ ]:
# Import basic packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

## TensorFlow Import and GPU Check

In this section, we import the **TensorFlow** package, which is one of the most widely used deep learning frameworks along with **PyTorch**. We will use it to build and train neural networks. We also check the **TensorFlow version** to ensure compatibility with our code.  

Next, we verify whether a **GPU is allocated** in this Colab environment. A GPU significantly accelerates deep learning model training compared to using only a CPU. If no GPU is found, the code will raise an error.

> Make sure that the output shows a GPU device (e.g., `/device:GPU:0`) to confirm that your Colab session is using GPU resources.

> You can check or change the GPU setting in this Colab notebook by going to the top menu:  
`Runtime` → `Change runtime type`, and then selecting **GPU** under *Hardware accelerator*.



In [ ]:
# Import the 'Tensorflow' pakage
import tensorflow as tf
from tensorflow import keras

# Check the version of tensorflow
print(tf.__version__)

In [ ]:
# Check if a GPU (in Google server) is allocated
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    raise SystemError('GPU device not found')

print('Found GPU at: {}'.format(device_name))

.

.

## Load the SELECTED (Top 30) Feature Dataset
* Result from the DA3-2

In [ ]:
FeatureSelected = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SavedFiles/FeatureSelected.csv', header=None)
FeatureSelected = FeatureSelected.T
FeatureSelected.shape

#### **Why Do We Standardize Features?**

Before training a machine learning model, it is important to **standardize the feature values**.  
Different features may have very different scales (e.g., one feature in the range of 0–1, another in the range of thousands).  

If we do not standardize, features with larger scales may dominate the training process, causing the model to perform poorly.  
By standardizing, we ensure that all features contribute **equally** to the learning process, which improves both the **convergence speed** and the **accuracy** of the model.


In [ ]:
# Standardize feature values
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

FeatureSelected_std = StandardScaler().fit_transform(FeatureSelected)
FeatureSelected_std.shape

## Splitting Training and Test Data

To evaluate the performance of a machine learning model, we need to split the dataset into two parts:  
- **Training set**: used to train the model.  
- **Test set**: used to evaluate the model’s performance on unseen data.  

Here, we divide the dataset with an **80:20 ratio** (80% training, 20% testing).  
This ensures that the model learns from the majority of the data while still being validated on a separate portion to check its generalization ability.  

The `train_test_split` function from `scikit-learn` is used to randomly divide the data into training and test sets.  
A fixed `random_state` is specified to guarantee reproducibility of the results.


In [ ]:
# Number of data for each condition: 180
NoOfData   = int(FeatureSelected_std.shape[0]/2)

NormalSet   = FeatureSelected_std[:NoOfData , :]
AbnormalSet = FeatureSelected_std[NoOfData: , :]

NormalSet.shape, AbnormalSet.shape

In [ ]:
from sklearn.model_selection    import train_test_split

# Designate test data ratio
TestData_Ratio = 0.2

TrainData_Nor, TestData_Nor = train_test_split(NormalSet  , test_size=TestData_Ratio, random_state=777)
TrainData_Abn, TestData_Abn = train_test_split(AbnormalSet, test_size=TestData_Ratio, random_state=777)

print(TrainData_Nor.shape, TestData_Nor.shape)
print(TrainData_Abn.shape, TestData_Abn.shape)

## Data Labeling with One-Hot Encoding

To train a classification model, we need to provide **labels** that indicate whether each sample is *Normal* or *Abnormal*.  

Here, we use **one-hot encoding**, a common method to represent categorical labels:  
- `[1, 0]` → Normal  
- `[0, 1]` → Abnormal  

This encoding makes it easier for the neural network to process class information during training, as it transforms categorical outputs into a numerical format that the model can learn from effectively.  

The labels are created using `np.zeros` and `np.ones` for both the training and test datasets, matching the number of samples in each group.


In [ ]:
TrainLabel_Nor = np.zeros((TrainData_Nor.shape[0],2))
TrainLabel_Abn = np.ones( (TrainData_Abn.shape[0],2))
TestLabel_Nor  = np.zeros((TestData_Nor.shape[0],2))
TestLabel_Abn  = np.ones( (TestData_Abn.shape[0],2))

TrainLabel_Nor[:,0] = 1  # [1,0]: Normal
TrainLabel_Abn[:,0] = 0  # [0,1]: Abnormal
TestLabel_Nor[:,0]  = 1  # [1,0]: Normal
TestLabel_Abn[:,0]  = 0  # [0,1]: Abnormal

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

In [ ]:
TestLabel_Nor

## Data and Label Preparation

At this stage, we combine the previously separated **Normal** and **Abnormal** subsets into unified datasets:  
- `TrainData` and `TestData` hold all feature values.  
- `TrainLabel` and `TestLabel` hold the corresponding one-hot encoded labels.  

Now, both the input data and the target labels are fully prepared for training and evaluating the machine learning model.

In [ ]:
TrainData  = np.concatenate([TrainData_Nor , TrainData_Abn ], axis=0)
TestData   = np.concatenate([TestData_Nor  , TestData_Abn  ], axis=0)
TrainLabel = np.concatenate([TrainLabel_Nor, TrainLabel_Abn], axis=0)
TestLabel  = np.concatenate([TestLabel_Nor , TestLabel_Abn ], axis=0)

print(TrainData.shape,  TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

.

.

.

.

.

## Setting hyperparameters for training MLP (Multi-Layer Perceptron)

Before training, we first define the **hyperparameters** that control the learning process:

- **learningRate**: step size used by the optimizer when updating weights.  
  A smaller value leads to slower but more stable learning, while a larger value can speed up training but may cause instability.  

- **noOfNeuron**: number of neurons (units) in each hidden layer.  
  More neurons can capture complex patterns but also increase computational cost and risk of overfitting.  

- **Epoch**: number of times the entire dataset is passed through the network during training.  
  More epochs allow the model to learn better, but too many can lead to overfitting.  

In [ ]:
learningRate  = 0.0001
noOfNeuron    = 16
Epoch         = 200

## Designing an MLP (ANN) architecture (based on Keras)

The MLP (Multi-Layer Perceptron) is implemented using **Keras Sequential API**:
- **Input Layer**: matches the number of features in the dataset.  
- **Hidden Layers**: use ReLU activation to capture non-linear relationships.  
- **Output Layer**: uses Softmax activation with 2 neurons (Normal / Abnormal classification).  

The model is compiled with:
- **Optimizer**: Adam (adaptive learning rate optimization algorithm).  
- **Loss Function**: Categorical Crossentropy (common for multi-class classification).  
- **Metric**: Accuracy (to measure classification performance).  

- Types of Activation Functions: https://keras.io/api/layers/activations/

- Types of Optimization Algorithms: https://keras.io/api/optimizers/

- Types of Loss Functions (for Classification)  https://keras.io/api/losses/probabilistic_losses/

In [ ]:
def MLP_model(input_data):
    keras.backend.clear_session() # clearing the Keras backend session (initiating variables)

    model = keras.Sequential()
    model.add(keras.layers.InputLayer(input_shape = (input_data.shape[1],) ))                                      # Input  Layer
    model.add(keras.layers.Dense(units = noOfNeuron, activation = keras.activations.relu,    name = 'Hidden1'))    # Hidden Layer 1
    model.add(keras.layers.Dense(units = noOfNeuron, activation = keras.activations.relu,    name = 'Hidden2'))    # Hidden Layer 2
    model.add(keras.layers.Dense(units = 2,          activation = keras.activations.softmax, name = 'Output'))     # Output Layer

    model.compile(optimizer = keras.optimizers.Adam(learning_rate = learningRate), # Optimization algorithm
                  loss = keras.losses.CategoricalCrossentropy(),                   # Loss function (objective function of Optimization)
                  metrics = ['accuracy'])                                          # Metrics to measure during the training process
    return model

In [ ]:
# Check the model architecture and the number of parameters
MLP = MLP_model(TrainData)
MLP.summary()

In [ ]:
# Check the parameter shape for each layer
for i in range(len(MLP.get_weights())):
    print(MLP.get_weights()[i].shape)

### Training and Evaluating the MLP (ANN) Model

With the model architecture defined, we now train the model using the prepared training data:  
- The model learns patterns by minimizing the **loss function** over multiple epochs.  
- Training history (loss and accuracy) is recorded for later visualization.  

In [ ]:
tf.random.set_seed(777) # Not necessarily required

# Model traning and validation
TraingHistory  = MLP.fit(TrainData, TrainLabel, epochs=Epoch, verbose=1)

After training, we evaluate the model on the **test dataset**, which was not seen during training.  
- **Loss**: indicates how far the model’s predictions are from the true labels (lower is better).  
- **Accuracy**: proportion of correctly classified samples (closer to 1.0 or 100% is better).

In [ ]:
# Evaluation result for test data (not trained)
Loss, Accuracy = MLP.evaluate(TestData,  TestLabel, verbose=0)
Loss, Accuracy # The closer the Loss is to 0 and the closer the accuracy is to 1 (100%), the better.

Finally, we plot the training curves (loss and accuracy vs. epoch) to check the learning progress.  
- A smooth decrease in loss and an increase in accuracy suggest that the model is learning well.  
- If the curves diverge (e.g., accuracy improves on training data but not on test data), it may indicate **overfitting**.

In [ ]:
# Check the training process (Loss, Accuracy)

fig, loss_ax = plt.subplots(figsize=(8,6))
acc_ax = loss_ax.twinx()

loss_ax.plot(TraingHistory.history['loss'], label='train loss', c = 'tab:red')
loss_ax.set_xlabel('epoch', fontsize=15)
loss_ax.set_ylabel('loss', fontsize=15)
loss_ax.legend(loc='upper left', fontsize=15)

acc_ax.plot(TraingHistory.history['accuracy'], label='train acc', c = 'tab:blue')
acc_ax.set_ylabel('accuracy', fontsize=15)
acc_ax.legend(loc='lower left', fontsize=15)

plt.show()

### [Tip] Using a Callback to Monitor Training Progress

During training, `verbose=1` prints the result at **every epoch**, which can produce a very long output when training for many epochs.  

To make the log easier to read, we can define a **custom callback function** that prints the training accuracy only at specific intervals (e.g., every 20 epochs).  

- **Callback**: a function that Keras automatically calls at certain points during training (e.g., at the end of each epoch).  
- In this example, the callback checks whether the current epoch number is a multiple of 20 and prints the training accuracy.  

This approach provides a **cleaner summary of training progress** while still allowing us to monitor the model’s performance at regular intervals.


In [ ]:
# Define the callback function
EpochForPrint = 20

class CheckProcess(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        keras.callbacks.Callback()
        if epoch%EpochForPrint == 0:
            print("{} Epochs Train Acc. : {:.2f}%  ".format(epoch, logs["accuracy"]*100))

In [ ]:
MLP_2 = MLP_model(TrainData)
hist = MLP_2.fit(TrainData, TrainLabel, epochs=Epoch, verbose=0, callbacks=[CheckProcess()])

print('Final Train Accuracy : {:.2f}%'.format(hist.history['accuracy'][-1]*100))

.

.

Save ML model (ANN) as a file

In [ ]:
MLP.save('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/MLP_model.keras')

Load the saved ML model (MLP) and test

In [ ]:
LoadedModel = keras.models.load_model('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/MLP_model.keras')

Loss, Accuracy = LoadedModel.evaluate(TestData, TestLabel, verbose=0)
print('[Performance of ANN model] \n')
print('Accuracy : {:.2f}%'.format(Accuracy*100))

In [ ]:
# Predicted result
Predicted = LoadedModel.predict(TestData)
pd.DataFrame(Predicted)

.

.

.

## Model Evaluation: Confusion Matrix and Metrics

After training, we need to evaluate how well the model performs on the test data.  
For binary classification (Normal vs. Abnormal), we use a **confusion matrix** and four common evaluation metrics.


In [ ]:
# Convert TestLabel and Predicted into single-column vectors for evaluations
TestLabel_rev = np.argmax(TestLabel, axis=1)
Predicted_rev = np.argmax(Predicted, axis=1)

Predicted_rev

#### Confusion Matrix
A confusion matrix summarizes the number of correct and incorrect predictions:
- **True Positive (TP)**: Abnormal samples correctly predicted as Abnormal.  
- **True Negative (TN)**: Normal samples correctly predicted as Normal.  
- **False Positive (FP)**: Normal samples incorrectly predicted as Abnormal.  
- **False Negative (FN)**: Abnormal samples incorrectly predicted as Normal.  

This gives us a full picture of the model’s strengths and weaknesses, beyond just accuracy.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(TestLabel_rev, Predicted_rev)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix of the MLP Model")
plt.show()

#### Evaluation Metrics
- **Accuracy** = (TP + TN) / (Total samples)  
  → Overall proportion of correct predictions.  

- **Precision** = TP / (TP + FP)  
  → Of all samples predicted as Abnormal, how many were actually Abnormal?  
  (Focuses on reducing false alarms.)  

- **Recall** = TP / (TP + FN)  
  → Of all actual Abnormal samples, how many did the model correctly detect?  
  (Focuses on reducing missed detections.)  

- **F1-score** = 2 × (Precision × Recall) / (Precision + Recall)  
  → Harmonic mean of Precision and Recall, useful when the dataset is imbalanced.  

---

By examining these metrics together, we can better understand model performance:  
- High accuracy alone is not enough if the dataset is imbalanced.  
- Precision and Recall trade off with each other, and the F1-score balances both.  

In [ ]:
from sklearn import metrics

# Calculate the evaluation metrics
accuracy  = metrics.accuracy_score(TestLabel_rev, Predicted_rev)
precision = metrics.precision_score(TestLabel_rev, Predicted_rev)
recall    = metrics.recall_score(TestLabel_rev, Predicted_rev)
f1_score  = metrics.f1_score(TestLabel_rev, Predicted_rev)

# Print the evaluation metrics
print(f"MLP Model Evaluation:\n")
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1 Score : {f1_score:.2f}")